# 01 · Speculative Decoding — 2-3x Faster Generation

> **Module:** 10 – Advanced Inference Techniques  
> **Objective:** Understand and implement speculative decoding, the technique that powers fast inference in modern LLM serving systems.

---

## 🎯 Learning Objectives

1. Identify the latency bottleneck in autoregressive generation
2. Derive the speculative decoding algorithm and its acceptance criterion
3. Implement the draft-verify loop from scratch
4. Prove that speculative decoding preserves the target distribution exactly
5. Understand when speculative decoding works well (and when it doesn't)

---

## 1. The Latency Problem

In LLM inference, generating 100 tokens requires 100 **sequential** forward passes through the large model.

```
LLaMA-3-70B on A100:
  Prefill:  ~100ms (parallel over prompt tokens)
  Decode:   ~30ms per token
  
  → 100 token response = 100 × 30ms = 3 seconds decode time
```

**Key insight**: Modern GPUs are memory-bandwidth bound during decode, not compute bound.
- A 70B model generates 1 token per forward pass
- The GPU can actually do much more compute per memory load
- We're leaving compute on the table!

**Speculative decoding** exploits this by:
1. Using a **small, fast draft model** to propose `γ` tokens cheaply
2. Using the **large target model** to verify all `γ` tokens in ONE parallel forward pass
3. Accepting verified tokens, rejecting bad ones

---

## 2. The Algorithm

```
SPECULATIVE DECODING ALGORITHM
═══════════════════════════════

Input: target model M_q, draft model M_p, prompt x, γ (lookahead length)

Repeat:
  1. DRAFT PHASE (small model):
     For i = 1..γ:
       Sample x̃_i ~ M_p(·| x, x̃_1..x̃_{i-1})   ← autoregressive, cheap

  2. VERIFY PHASE (large model):
     Compute q_i = M_q(x̃_i | x, x̃_1..x̃_{i-1})  ← ONE parallel pass for all γ!
     Compute p_i = M_p(x̃_i | x, x̃_1..x̃_{i-1})

  3. ACCEPT/REJECT:
     For i = 1..γ:
       If uniform() < min(1, q_i / p_i):         ← acceptance criterion
         Accept x̃_i
       Else:
         Reject x̃_i and all subsequent
         Sample a bonus token from adjusted distribution
         Break

     If all γ accepted:
       Sample one more token from M_q bonus

Key property: The accepted tokens are distributed EXACTLY as M_q would produce.
              → Speculative decoding is lossless! ✅
```

In [ ]:
# ── Setup ──────────────────────────────────────────────────────────────────
!pip install transformers torch -q

import torch
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import List, Tuple, Optional
from dataclasses import dataclass

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

## 3. Acceptance Criterion: Proof of Correctness

Why does this acceptance rule `min(1, q/p)` preserve the target distribution?

The probability that token `x` is ultimately emitted:
```
P(emit x) = P(accept) · p(x) + P(reject bonus) · adjusted_q(x)
          = min(1, q(x)/p(x)) · p(x) + (correction term)
          = q(x)   ← exact target distribution!
```

In [ ]:
def speculative_sample_one_step(
    q_probs: torch.Tensor,  # target model probabilities (V,)
    p_probs: torch.Tensor,  # draft model probabilities (V,)
    draft_token: int,
) -> Tuple[Optional[int], bool]:
    """
    Apply the speculative decoding acceptance criterion for ONE token.
    
    Returns: (token_id, accepted)
      - If accepted: returns (draft_token, True)
      - If rejected: samples from adjusted q, returns (new_token, False)
    """
    q_x = q_probs[draft_token].item()
    p_x = p_probs[draft_token].item()

    # Acceptance probability
    accept_prob = min(1.0, q_x / (p_x + 1e-9))

    if torch.rand(1).item() < accept_prob:
        # Accept the draft token
        return draft_token, True
    else:
        # Reject — sample from adjusted distribution: max(0, q - p) / Z
        adjusted = torch.clamp(q_probs - p_probs, min=0.0)
        Z = adjusted.sum()
        if Z < 1e-9:
            # Fallback: sample from q directly
            new_token = torch.multinomial(q_probs, 1).item()
        else:
            new_token = torch.multinomial(adjusted / Z, 1).item()
        return new_token, False


# Verify the distribution is preserved
V = 10  # tiny vocabulary for illustration
q = F.softmax(torch.randn(V), dim=0)  # target distribution
p = F.softmax(torch.randn(V), dim=0)  # draft distribution

N_SAMPLES = 100_000
counts = torch.zeros(V)

for _ in range(N_SAMPLES):
    # Draft: sample from p
    draft = torch.multinomial(p, 1).item()
    # Speculative accept/reject
    token, _ = speculative_sample_one_step(q, p, draft)
    counts[token] += 1

empirical = counts / N_SAMPLES

# Compare empirical vs target
fig, ax = plt.subplots(figsize=(10, 4))
x = range(V)
ax.bar([i - 0.2 for i in x], q.numpy(), width=0.35, label='Target q', alpha=0.8, color='steelblue')
ax.bar([i + 0.2 for i in x], empirical.numpy(), width=0.35, label='Empirical (spec. decoding)', alpha=0.8, color='tomato')
ax.set_title('Distribution Preservation Verification')
ax.set_xlabel('Token ID')
ax.set_ylabel('Probability')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/10_spec_decoding_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

kl = (q * (q / (empirical + 1e-9)).log()).sum().item()
print(f"KL(q || empirical) = {kl:.6f}  (should be ~0 for exact match)")

## 4. Full Speculative Decoding Loop

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

@dataclass
class SpecDecodeStats:
    total_tokens: int = 0
    accepted_tokens: int = 0
    target_calls: int = 0
    draft_calls: int = 0

    @property
    def acceptance_rate(self) -> float:
        return self.accepted_tokens / max(self.total_tokens, 1)

    @property
    def avg_tokens_per_target_call(self) -> float:
        return self.total_tokens / max(self.target_calls, 1)


class SpeculativeDecoder:
    """
    Implements speculative decoding using a large target model
    and a small draft model.
    
    For real deployment:
      - Target: LLaMA-3.1-70B
      - Draft: LLaMA-3.2-1B (same tokenizer!)
      
    Key constraint: target and draft MUST use the same tokenizer.
    """

    def __init__(
        self,
        target_model,
        draft_model,
        tokenizer,
        gamma: int = 4,  # draft tokens per step
        temperature: float = 1.0,
    ):
        self.target = target_model
        self.draft = draft_model
        self.tokenizer = tokenizer
        self.gamma = gamma
        self.temperature = temperature

    @torch.no_grad()
    def _get_probs(
        self,
        model,
        input_ids: torch.Tensor
    ) -> torch.Tensor:
        """Get next-token probability distribution."""
        logits = model(input_ids).logits[:, -1, :]
        if self.temperature != 1.0:
            logits = logits / self.temperature
        return F.softmax(logits, dim=-1).squeeze(0)  # (V,)

    @torch.no_grad()
    def _get_probs_batch(
        self,
        model,
        input_ids: torch.Tensor  # (1, seq_len + gamma)
    ) -> torch.Tensor:
        """
        Get probabilities for multiple positions in ONE forward pass.
        This is the key efficiency win in speculative decoding.
        Returns: (gamma + 1, V)
        """
        logits = model(input_ids).logits[0, -(self.gamma + 1):, :]  # last gamma+1 positions
        if self.temperature != 1.0:
            logits = logits / self.temperature
        return F.softmax(logits, dim=-1)  # (gamma+1, V)

    @torch.no_grad()
    def generate(
        self,
        input_ids: torch.Tensor,
        max_new_tokens: int = 50,
    ) -> Tuple[torch.Tensor, SpecDecodeStats]:
        """Generate tokens using speculative decoding."""
        stats = SpecDecodeStats()
        tokens = input_ids.clone()

        while stats.total_tokens < max_new_tokens:
            # ── DRAFT PHASE ──────────────────────────────────────
            # Draft model generates gamma tokens autoregressively
            draft_tokens = []
            draft_probs = []  # p(x̃_i | context)

            current = tokens.clone()
            for _ in range(self.gamma):
                p = self._get_probs(self.draft, current)  # (V,)
                x = torch.multinomial(p, 1).item()
                draft_tokens.append(x)
                draft_probs.append(p[x].item())
                current = torch.cat([current, torch.tensor([[x]])], dim=-1)
                stats.draft_calls += 1

            # ── VERIFY PHASE ─────────────────────────────────────
            # Target model processes prompt + all draft tokens in ONE pass
            verify_input = torch.cat(
                [tokens, torch.tensor([draft_tokens])], dim=-1
            )
            target_probs = self._get_probs_batch(self.target, verify_input)  # (gamma+1, V)
            stats.target_calls += 1

            # ── ACCEPT/REJECT PHASE ──────────────────────────────
            n_accepted = 0
            for i, (draft_tok, p_x) in enumerate(zip(draft_tokens, draft_probs)):
                q_x = target_probs[i][draft_tok].item()
                accept_prob = min(1.0, q_x / (p_x + 1e-9))

                if torch.rand(1).item() < accept_prob:
                    # Accept
                    tokens = torch.cat([tokens, torch.tensor([[draft_tok]])], dim=-1)
                    stats.accepted_tokens += 1
                    n_accepted += 1
                    stats.total_tokens += 1
                else:
                    # Reject — sample from adjusted distribution
                    adjusted = torch.clamp(target_probs[i] - torch.full_like(target_probs[i], p_x), min=0)
                    Z = adjusted.sum()
                    if Z > 1e-9:
                        bonus = torch.multinomial(adjusted / Z, 1).item()
                    else:
                        bonus = torch.multinomial(target_probs[i], 1).item()
                    tokens = torch.cat([tokens, torch.tensor([[bonus]])], dim=-1)
                    stats.total_tokens += 1
                    break
            else:
                # All gamma accepted — add one bonus token from target
                bonus = torch.multinomial(target_probs[self.gamma], 1).item()
                tokens = torch.cat([tokens, torch.tensor([[bonus]])], dim=-1)
                stats.total_tokens += 1

            # Check for EOS
            if tokens[0, -1].item() == self.tokenizer.eos_token_id:
                break

        return tokens, stats

## 5. Demo with Real Models

In [ ]:
# For demonstration, we use GPT-2 as both target and a tiny variant as draft.
# In production: target=LLaMA-70B, draft=LLaMA-1B

print("Loading models...")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
target_model = AutoModelForCausalLM.from_pretrained("gpt2").eval()           # 117M (target)
draft_model  = AutoModelForCausalLM.from_pretrained("distilgpt2").eval()    # 82M (draft)

# Move to device
target_model = target_model.to(device)
draft_model  = draft_model.to(device)

prompt = "The future of artificial intelligence is"
input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)

# ── Baseline: standard greedy decoding ───────────────────────────────────
t0 = time.perf_counter()
with torch.no_grad():
    baseline_out = target_model.generate(
        input_ids, max_new_tokens=50, do_sample=True, temperature=1.0
    )
baseline_time = time.perf_counter() - t0
baseline_text = tokenizer.decode(baseline_out[0], skip_special_tokens=True)

# ── Speculative decoding ─────────────────────────────────────────────────
spec_decoder = SpeculativeDecoder(target_model, draft_model, tokenizer, gamma=4)

t0 = time.perf_counter()
spec_out, stats = spec_decoder.generate(input_ids, max_new_tokens=50)
spec_time = time.perf_counter() - t0
spec_text = tokenizer.decode(spec_out[0], skip_special_tokens=True)

n_new = spec_out.shape[1] - input_ids.shape[1]

print("\n" + "="*60)
print("BASELINE OUTPUT:")
print(baseline_text)
print(f"Time: {baseline_time*1000:.0f}ms")

print("\n" + "="*60)
print("SPECULATIVE DECODING OUTPUT:")
print(spec_text)
print(f"Time: {spec_time*1000:.0f}ms")
print(f"Speedup: {baseline_time/spec_time:.2f}x")

print("\n" + "="*60)
print("STATS:")
print(f"  Tokens generated:           {stats.total_tokens}")
print(f"  Accepted draft tokens:      {stats.accepted_tokens}")
print(f"  Acceptance rate:            {stats.acceptance_rate:.1%}")
print(f"  Target model calls:         {stats.target_calls}")
print(f"  Avg tokens per target call: {stats.avg_tokens_per_target_call:.2f}")

## 6. Experiment: Gamma vs Speedup

In [ ]:
# How does the lookahead gamma affect speedup and acceptance rate?
gammas = [1, 2, 4, 6, 8, 12]
results = []

for gamma in gammas:
    decoder = SpeculativeDecoder(target_model, draft_model, tokenizer, gamma=gamma)
    all_stats = []
    
    for _ in range(5):  # average over 5 runs
        _, stats = decoder.generate(input_ids.clone(), max_new_tokens=30)
        all_stats.append(stats)
    
    avg_acceptance = np.mean([s.acceptance_rate for s in all_stats])
    avg_tpt = np.mean([s.avg_tokens_per_target_call for s in all_stats])
    results.append({
        'gamma': gamma,
        'acceptance_rate': avg_acceptance,
        'tokens_per_call': avg_tpt,
    })
    print(f"γ={gamma:2d} | acceptance={avg_acceptance:.1%} | tokens/target_call={avg_tpt:.2f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

ax = axes[0]
ax.plot([r['gamma'] for r in results], [r['acceptance_rate'] for r in results],
        'o-', color='steelblue', linewidth=2, markersize=8)
ax.set_xlabel('Gamma (draft length)')
ax.set_ylabel('Average Acceptance Rate')
ax.set_title('Acceptance Rate vs Gamma')
ax.grid(alpha=0.3)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'{x:.0%}'))

ax = axes[1]
ax.plot([r['gamma'] for r in results], [r['tokens_per_call'] for r in results],
        'o-', color='darkorange', linewidth=2, markersize=8)
ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Baseline (1 token/call)')
ax.set_xlabel('Gamma (draft length)')
ax.set_ylabel('Tokens per Target Model Call')
ax.set_title('Throughput vs Gamma')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../figures/10_spec_decoding_gamma.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Engineering Notes

### When Does Speculative Decoding Help?

| Scenario | Speedup | Why |
|----------|---------|-----|
| Greedy / temperature=0 | **3-4x** | Draft almost always matches target |
| Low temperature (0.3-0.7) | **2-3x** | High acceptance rate |
| High temperature (>1.0) | **1.2-1.5x** | Draft often rejected |
| Long context, short output | **Minimal** | Prefill dominates |

### Production Considerations

```python
# Draft model must:
# 1. Use SAME tokenizer as target (same vocab!)
# 2. Be 10-20x smaller than target
# 3. Be architecturally similar (both decoder-only, same vocab)

# Good pairings:
# LLaMA-3.1-70B (target) + LLaMA-3.2-1B (draft)   → 2.5x speedup
# LLaMA-3.1-8B (target)  + LLaMA-3.2-1B (draft)   → 1.8x speedup  
# GPT-4 (target)         + GPT-3.5-turbo (draft)   → not public API
```

### Variants

| Method | Description | Speedup |
|--------|-------------|--------|
| **Standard Spec Dec** | External draft model | 2-3x |
| **EAGLE** | Draft head trained on same model internals | 3-4x |
| **Medusa** | Multiple draft heads in parallel | 2-3x |
| **Self-speculation** | Use early exits as draft | 1.5-2x |

## 8. Exercises

1. **Temperature Sweep**: Run speculative decoding with temperatures [0, 0.3, 0.7, 1.0, 1.5]. Plot acceptance rate vs temperature.

2. **Optimal Gamma**: For a fixed draft model quality, derive the optimal γ analytically as a function of acceptance rate α. Then verify empirically.

3. **Self-Drafting**: Implement a simple version of EAGLE where the draft is produced by the same model's first few layers only (early exit). Compare vs external draft.

4. **Batch Speculative Decoding**: Extend the implementation to handle a batch of sequences (batch_size > 1). What complications arise?

5. **Benchmark**: Using vLLM's built-in speculative decoding, compare throughput with and without speculative decoding on a real workload.

---

## 9. References

- [Leviathan et al. (2023) — Fast Inference from Transformers via Speculative Decoding](https://arxiv.org/abs/2211.17192)
- [Chen et al. (2023) — Accelerating Large Language Model Decoding with Speculative Sampling](https://arxiv.org/abs/2302.01318)
- [Li et al. (2024) — EAGLE: Speculative Sampling Requires Rethinking Feature Uncertainty](https://arxiv.org/abs/2401.15077)
- [vLLM Speculative Decoding Docs](https://docs.vllm.ai/en/latest/models/spec_decode.html)